# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. It provides hands-on examples for accessing records, fields, and conducting exploratory data analysis (EDA).

### Dataset Source
The dataset is defined by a Croissant schema and accessible at the following URL:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (do not subscript; use attributes and dir/vars for details)
print("Dataset Name: ", dataset.metadata.name)
print("Version: ", getattr(dataset.metadata, 'version', 'N/A'))
print("Published: ", getattr(dataset.metadata, 'datePublished', 'N/A'))
print("Description:\n", dataset.metadata.description)


## 2. Data Overview
Let's examine available **record sets** and their fields (columns) via their `@id`s.

The function below collects all record sets present in the Croissant schema and prints their `@id` and available fields or columns.

In [ ]:
# List all record sets, fields, and columns with their @id
from pprint import pprint

def explore_record_sets(ds):
    print("Available Record Sets and Fields/Columns by @id:")
    record_sets = getattr(ds.metadata, "recordSets", None)
    if not record_sets:
        # Some schemas use 'recordSet' (singular or plural); try both
        record_sets = getattr(ds.metadata, "recordSet", None)
    if not record_sets:
        # Alternatively, find all keys that are record sets
        record_sets = [rs for rs in vars(ds.metadata).values() if hasattr(rs, 'fields') or hasattr(rs, 'columns')]
    # Standardize to a list
    try:
        record_sets = list(record_sets)
    except Exception:
        record_sets = [record_sets]
    record_set_ids = []
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        if not rs_id and hasattr(rs, '_id'):
            rs_id = getattr(rs, '_id', None)
        print(f"- RecordSet @id: {rs_id}")
        record_set_ids.append(rs_id)
        for field_name in ['fields', 'columns', 'field', 'column']:
            components = getattr(rs, field_name, None)
            if components:
                if not isinstance(components, list):
                    components = [components]
                for fld in components:
                    field_id = getattr(fld, '@id', None) or getattr(fld, '_id', None) or str(fld)
                    fname = getattr(fld, 'name', None)
                    print(f"    - Field/Column: {fname} (@id: {field_id})")
    return record_set_ids
        
# Extract record set ids for later use
record_set_ids = []
record_sets = getattr(dataset.metadata, 'recordSet', [])
if not record_sets:
    # Some schemas may define hasPart as entries for record sets
    record_sets = getattr(dataset.metadata, 'hasPart', [])

# Print details if present, otherwise try to infer
if record_sets:
    print("Record sets found via 'recordSet':")
    for recset in record_sets:
        recsetid = getattr(recset, '@id', None)
        print(f"- {recsetid}")
        record_set_ids.append(recsetid)
else:
    # Try to infer record sets by exploring record_set_ids from dataset (via mlcroissant v0.6+)
    print("Inferring record set IDs by listing available data targets...")
    # If mlcroissant exposes dataset.metadata.data_files or dataset.target_ids, use that
    try:
        print("Available targets:", dataset.target_ids)
        record_set_ids = dataset.target_ids
    except Exception:
        print("Could not infer record_set_ids directly. Proceed by inspecting dataset.metadata.")
if not record_set_ids:
    # Default: try the default data target for typical datasets
    record_set_ids = ['cr:recordSet']

# For each record set, print sample records using the API
for rs_id in record_set_ids:
    print(f"\nSample records from record set @id='{rs_id}':")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            pprint(rec)
            if i >= 2: break
    except Exception as e:
        print(f"  Could not fetch records for {rs_id}: {e}")


## 3. Data Extraction
Now we extract data from each record set into pandas DataFrames, using the `@id` for each set.

We demonstrate this with the main record set(s) detected in the overview.

In [ ]:
# List of record set ids obtained from above
record_sets = record_set_ids  # e.g., ['cr:recordSet', ...]
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded DataFrame for {record_set} with shape {df.shape}.")
        else:
            print(f"No records found for {record_set}.")
    except Exception as e:
        print(f"Error extracting records for {record_set}: {e}")

# Pick the largest dataframe (or the first one) for demonstration
if dataframes:
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Fields (columns) for record set @id='{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    main_rs_id = None
    print("No DataFrames loaded.")


## 4. Exploratory Data Analysis (EDA)
We demonstrate EDA for one record set. Numeric, categorical, and text columns can be filtered using their `@id`/column name.

- We'll select a numeric field (e.g., protein abundance, molecular weight, peptide counts).
- Filter, normalize, and group the data (with robust handling if fields are unavailable).

In [ ]:
import numpy as np

if main_rs_id is not None and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    # Choose a likely numeric field (prioritize typical proteomics fields)
    possible_numeric_fields = [
        'Molecular weight', 'MW', 'molecular_weight', 'PeptideCount', 'Coverage', 'pI',
        'Normalized abundance', 'Abundance', 'abundance', 'Peptide_Count', 'Unique_Peptides',
    ]
    matched_numeric = None
    for col in df.columns:
        for candidate in possible_numeric_fields:
            if candidate.lower() in col.lower():
                matched_numeric = col
                break
        if matched_numeric:
            break
    if not matched_numeric:
        # Fallback: find the first numeric column
        for col in df.select_dtypes(include=[np.number]).columns:
            if not col.startswith('_'):
                matched_numeric = col
                break
    print(f"Selected numeric field: '{matched_numeric}'")
    # Demonstrate filtering: only keep records with value above threshold
    if matched_numeric and matched_numeric in df.columns:
        threshold = float(df[matched_numeric].quantile(0.5)) if np.issubdtype(df[matched_numeric].dtype, np.number) else 10
        filtered_df = df[df[matched_numeric] > threshold].copy()
        print(f"Filtered records where '{matched_numeric}' > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the selected column
        filtered_df[f"{matched_numeric}_normalized"] = (
            filtered_df[matched_numeric] - filtered_df[matched_numeric].mean()
        ) / (filtered_df[matched_numeric].std() + 1e-9)
        print(f"Normalized '{matched_numeric}' for filtered records:")
        display(filtered_df[[matched_numeric, f"{matched_numeric}_normalized"]].head())

        # Try to group by a likely categorical field (e.g., 'Description', 'Sample', 'Experiment', etc.)
        possible_groupers = [
            'Sample', 'Group', 'Description', 'Condition', 'Replicate', 'ProteinID', 'Accession'
        ]
        group_field = None
        for candidate in possible_groupers:
            for col in df.columns:
                if candidate.lower() in col.lower():
                    group_field = col
                    break
            if group_field:
                break
        print(f"Selected group field: '{group_field}'")
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
else:
    print("No main DataFrame available for EDA.")


## 5. Visualization
Visualize the distribution of the selected numeric field and its normalization, and a possible grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and main_rs_id in dataframes and matched_numeric:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][matched_numeric].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{matched_numeric}' in {main_rs_id}")
    plt.xlabel(matched_numeric)
    plt.tight_layout()
    plt.show()

    # If normalised version exists
    if f"{matched_numeric}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{matched_numeric}_normalized"].dropna(), bins=30, kde=True, color='orange')
        plt.title(f"Normalized '{matched_numeric}' (Filtered)")
        plt.xlabel(f"{matched_numeric}_normalized")
        plt.tight_layout()
        plt.show()

    # Plot group means if grouped_df is available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 6))
        grouped_df[matched_numeric].sort_values(ascending=False).plot(kind='bar')
        plt.title(f"Mean '{matched_numeric}' by '{group_field}'")
        plt.ylabel(matched_numeric)
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No visualization can be generated.")


## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, explored record set and field structure using their `@id`, extracted data to DataFrames, applied common EDA steps (filter, normalize, group), and visualized core numeric features. This workflow provides a reproducible approach for FAIR datasets conforming to the Croissant schema and eases downstream analysis and machine learning workflows.